In [1]:
from datasets import load_dataset

ds = load_dataset("Genius-Society/Pima")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/614 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/77 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/77 [00:00<?, ? examples/s]

In [6]:
!pip install pennylane datasets
import pennylane as qml
from datasets import load_dataset
from pennylane import numpy as np
# Extraire les features (8 features)
X = np.array([[s['Pregnancies'], s['Glucose'], s['BloodPressure'], s['SkinThickness'],
               s['Insulin'], s['BMI'], s['DiabetesPedigreeFunction'], s['Age']]
              for s in ds['train']])
y = np.array([s['Outcome'] for s in ds['train']])

# Normalisation des features pour rotation (0 à pi)
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
X_angle = X_norm * np.pi  # angle entre 0 et pi

# Nombre de qubits = nombre de features
num_qubits = X.shape[1]


#  Définir le device quantique

dev = qml.device("default.qubit", wires=num_qubits)


# Définir le circuit variational

@qml.qnode(dev)
def variational_encoding(x, theta):
    """
    x     : features encodées en angles
    theta : paramètres entraînables
    """
    # Couche 1 : encoder les features + rotations entraînables
    for i in range(num_qubits):
        qml.RY(x[i], wires=i)        # encode la feature
        qml.RZ(theta[i], wires=i)    # rotation paramétrique entraînable

    # Entanglement
    for i in range(num_qubits - 1):
        qml.CNOT(wires=[i, i+1])

    # Couche 2 : re-uploading des features
    for i in range(num_qubits):
        qml.RY(x[i], wires=i)
        qml.RZ(theta[i + num_qubits], wires=i)

   # Retourne l'état complet (vecteur d'amplitudes)
    return qml.state()

#️ Initialiser les paramètres θ

theta_init = np.random.rand(num_qubits * 2)  # 2 couches, 8 qubits → 16 paramètres




In [3]:
# -------------------------------
# 4️⃣ Encodage d'un échantillon
# -------------------------------
sample = X_angle[0]
encoded_state = variational_encoding(sample, theta_init)

print("Etat quantique encodé (vecteur d'amplitudes) :")
print(encoded_state)

Etat quantique encodé (vecteur d'amplitudes) :
[ 6.38343330e-03+3.36051833e-03j  8.79003795e-03+1.66113340e-02j
  8.81497867e-03+7.85288400e-04j  1.38800912e-02-5.92788697e-02j
  3.43288263e-02+3.73771427e-02j -6.13542460e-03-3.00117414e-03j
 -7.28924861e-03+1.03794641e-02j -4.56931072e-02+4.05082614e-02j
  3.85080197e-02-2.72863712e-02j  2.52423619e-02+8.18159952e-03j
  1.57137137e-02-4.11256230e-03j -3.29114526e-02-3.35004816e-02j
  1.08207074e-01-1.21769681e-02j -1.51448748e-03+1.48239228e-02j
  1.45551467e-02+2.23269194e-02j  3.92334458e-03+8.11767446e-02j
  1.44971529e-02-7.32389776e-03j  1.51557322e-03+3.63631164e-03j
  4.03856371e-03+2.24110449e-03j  6.14097028e-03+4.97450607e-03j
  2.23997101e-02+4.12333895e-03j  1.15489989e-03+4.55183711e-03j
  2.76671926e-03+4.65043526e-03j -5.79366432e-03+6.27030461e-03j
 -6.61196760e-02+2.71531070e-02j -3.69221130e-02-1.49921694e-02j
 -2.25842462e-02+2.93025754e-03j  5.78015959e-02+5.26283205e-02j
 -1.66124100e-01-4.88197946e-03j  7.6768730

In [4]:
# 1. On transforme tout le dataset
X_variational = []

print("Calcul des états variationnels...")
for row in X_angle:
    # On récupère le vecteur d'état (state)
    state = variational_encoding(row, theta_init)
    # On utilise la magnitude au carré (probabilités) pour l'arbre classique
    X_variational.append(np.abs(state)**2)

X_variational = np.array(X_variational)
print(f"Nouvelle dimension : {X_variational.shape}") # (768, 256)

Calcul des états variationnels...
Nouvelle dimension : (614, 256)


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Split
X_train_v, X_test_v, y_train, y_test = train_test_split(X_variational, y, test_size=0.2, random_state=42)

# Entraînement
clf_variational = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_variational.fit(X_train_v, y_train)

# Score
accuracy = clf_variational.score(X_test_v, y_test)
print(f"Précision avec Variational Re-uploading (Random Theta) : {accuracy:.2%}")

Précision avec Variational Re-uploading (Random Theta) : 67.48%
